In [1]:
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(argparse))
suppressPackageStartupMessages(library(Seurat))

#####################
## Define settings ##
#####################
here::i_am("processing/1_create_seurat_rna.R")
source(here::here("settings.R"))
source(here::here("utils.R"))



here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/01_Eomes_RNA/code



In [2]:
opts$stages

[1] "E8.5"

In [3]:
args = list()
args$stages <- 'E8.5'
args$sce <- io$rna.sce
args$metadata <- paste0(io$basedir,"/results/rna/doublet_detection/sample_metadata_after_doublets.txt.gz")
args$atlas_stages = c('E6.0', 'E6.25', 'E6.5', 'E6.75', 'E7.0', 'E7.25', 'E7.5', 'E7.75', 'mixed_gastrulation','E8.0', 'E8.25', 'E8.5')
args$features <- 3000
args$npcs <- 40
args$n_neighbors = 30
args$min_dist = 0.3
args$test <- FALSE
args$vars.to.regress <- c("nFeature_RNA","mitochondrial_percent_RNA")
args$remove_ExE_cells <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/mapping/test")

In [4]:
dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)

In [5]:
sample_metadata <- fread(args$metadata) %>%
   .[pass_rnaQC==TRUE & doublet_call==FALSE & stage %in% args$stages]

if (args$remove_ExE_cells) {
  print("Removing ExE cells...")
  sample_metadata <- sample_metadata %>%
    .[!celltype.mapped_mnn%in%c("Visceral_endoderm","ExE_endoderm","ExE_ectoderm","Parietal_endoderm")]
}

In [6]:
###################
## Sanity checks ##
###################

stopifnot(args$colour_by %in% colnames(sample_metadata))
# stopifnot(unique(sample_metadata$celltype.mapped) %in% names(opts$celltype.colors))

 if (length(args$batch_correction)>0) {
   stopifnot(args$batch_correction%in%colnames(sample_metadata))
   if (length(unique(sample_metadata[[args$batch_correction]]))==1) {
     message(sprintf("There is a single level for %s, no batch correction applied",args$batch_correction))
     args$batch_correction <- NULL
   } else {
     library(batchelor)
   }
 }

 if (length(args$vars_to_regress)>0) {
  stopifnot(args$vars_to_regress%in%colnames(sample_metadata))
 }

In [7]:
###############
## Load data ##
###############

# Load RNA expression data as SingleCellExperiment object
sce_query <- load_SingleCellExperiment(args$sce, cells=sample_metadata$cell, normalise = TRUE)

# Add sample metadata as colData
colData(sce_query) <- sample_metadata %>% tibble::column_to_rownames("cell") %>% DataFrame


In [8]:
################
## Load atlas ##
################

# Load cell metadata
meta_atlas <- fread(io$rna.atlas.metadata) %>%
  .[stage%in%args$atlas_stages] %>%
  .[,sample:=factor(sample)] 

# Filter
if (isTRUE(args$test)) meta_atlas <- head(meta_atlas,n=1000)

# Load SingleCellExperiment
sce_atlas <- load_SingleCellExperiment(io$rna.atlas.sce, normalise = TRUE, cells = meta_atlas$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_atlas %>% .[cell%in%colnames(sce_atlas)] %>% setkey(cell) %>% .[colnames(sce_atlas)]
stopifnot(tmp$cell == colnames(sce_atlas))
colData(sce_atlas) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()

# Sanity cehcks
stopifnot(sum(is.na(rownames(sce_atlas)))==0)
stopifnot(sum(duplicated(rownames(sce_atlas)))==0)

In [9]:
#####################
## Define gene set ##
#####################

# Get gene metadata
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce_atlas)] %>%
  .[!duplicated(symbol)]

rownames(sce_atlas) = gene_metadata[match(rownames(sce_atlas), ens_id), symbol]

# Imprinted genes
imprint = gene_metadata[c(grep('maternally', gene_metadata$description),
                       grep('paternally', gene_metadata$description)), symbol]
#Other imprinted genes: 
#- Nnat (https://www.genecards.org/cgi-bin/carddisp.pl?gene=NNAT)
#- Grb10 (https://www.genecards.org/cgi-bin/carddisp.pl?gene=GRB10)

# Intersect genes
genes.intersect <- intersect(rownames(sce_query), rownames(sce_atlas))

# Filter some genes manually
genes.intersect <- genes.intersect[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",genes.intersect,invert=T)] # filter out non-informative genes
genes.intersect <- genes.intersect[grep("^Hbb|^Hba",genes.intersect,invert=T)] # test removing Haem genes 
genes.intersect <- genes.intersect[!genes.intersect %in% c(imprint, 'Grb10', 'Nnat')] # remove imprinted genes
genes.intersect <- genes.intersect[!genes.intersect %in% c("Xist", "Tsix")] # remove Xist & Tsix
genes.intersect <- genes.intersect[!genes.intersect=="tomato-td"] # remove tomato itself
genes.intersect <- genes.intersect[!genes.intersect %in% gene_metadata[chr=="chrY",symbol]] # no genes on y-chr 

# Subset SingleCellExperiment objects
sce_query  <- sce_query[genes.intersect,]
sce_atlas <- sce_atlas[genes.intersect,]

In [10]:
sce_query
sce_atlas

class: SingleCellExperiment 
dim: 16157 32368 
metadata(0):
assays(2): counts logcounts
rownames(16157): Xkr4 Rp1 ... Zfp950 Csf2ra
rowData names(0):
colnames(32368): SLX-20795_SITTA11_HKTG2DRXY#AAACCCAGTTCTCTAT-1
  SLX-20795_SITTA11_HKTG2DRXY#AAACGCTAGTGATTCC-1 ...
  SLX-20795_SITTH11_HKTG2DRXY#TTTGTTGCAATGAGCG-1
  SLX-20795_SITTH11_HKTG2DRXY#TTTGTTGCATATGCGT-1
colData names(13): sample barcode ... doublet_score doublet_call
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

class: SingleCellExperiment 
dim: 16157 116312 
metadata(1): log.exprs.offset
assays(2): counts logcounts
rownames(16157): Xkr4 Rp1 ... Zfp950 Csf2ra
rowData names(0):
colnames(116312): cell_1 cell_2 ... cell_139330 cell_139331
colData names(13): barcode sample ... index celltype_extended
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [11]:
seurat_query = as.Seurat(sce_query)
seurat_atlas = as.Seurat(sce_atlas)

In [12]:
seurat_atlas = RenameAssays(seurat_atlas, originalexp = 'RNA')

Renaming default assay from originalexp to RNA

Warning message:
“Cannot add objects with duplicate keys (offending key: originalexp_) setting key to original value 'rna_'”


In [24]:
seurat_query

An object of class Seurat 
16157 features across 32368 samples within 1 assay 
Active assay: RNA (16157 features, 0 variable features)

In [25]:
seurat_atlas

An object of class Seurat 
16157 features across 116312 samples within 1 assay 
Active assay: RNA (16157 features, 0 variable features)

In [26]:
# Multi core using future - built in to seurat
plan("multicore", workers = 24)
options(future.globals.maxSize = 50 * 1024 ^ 3) # for 50 Gb RAM

In [27]:
seurat.list = lapply(X = list(seurat_query, seurat_atlas), FUN = function(x) {
    x <- FindVariableFeatures(x, selection.method = "vst", nfeatures = args$features)
})

In [30]:
features <- SelectIntegrationFeatures(object.list = seurat.list)

In [31]:
VariableFeatures(seurat.list)

ERROR: Error in UseMethod(generic = "VariableFeatures", object = object): no applicable method for 'VariableFeatures' applied to an object of class "list"


In [29]:
head(features, 100)

[1] "Myl7"    "Actc1"   "Ttr"     "Myl4"    "Apoa1"   "Rbp4"    "Tnnt2"  
  [8] "Spink1"  "Acta2"   "Apom"    "Ankrd1"  "Acta1"   "Tnnc1"   "Myl3"   
 [15] "Nppa"    "Afp"     "Srgn"    "S100g"   "Rhox5"   "Ttn"     "Cdkn1c" 
 [22] "Nepn"    "Pyy"     "Fabp3"   "Apoe"    "Tnni1"   "Lyve1"   "Etv2"   
 [29] "Tpm1"    "Trh"     "Mt1"     "Lefty1"  "Myl2"    "Amn"     "Phlda2" 
 [36] "Crabp1"  "Ctsh"    "Myh7"    "Tagln"   "Pmp22"   "Lgals2"  "Cited1" 
 [43] "Sparc"   "Fam183b" "Lefty2"  "Mt2"     "Myh6"    "Spp2"    "Csrp3"  
 [50] "Apoa4"   "Cck"     "Krt18"   "Apob"    "Tcf15"   "Blvrb"   "Slc2a3" 
 [57] "Rhox6"   "Hoxaas3" "Cryab"   "Fbxl22"  "Sh3bgr"  "Krt8"    "Ptn"    
 [64] "Mesp1"   "Cldn5"   "Tdo2"    "Cdx1"    "Rhox9"   "Spin2c"  "Trap1a" 
 [71] "T"       "Cited4"  "Car2"    "Ripply2" "Mest"    "Crabp2"  "Cubn"   
 [78] "Meox1"   "Nebl"    "Car7"    "Hand1"   "Dlk1"    "Aldob"   "Nkx2-9" 
 [85] "Apoa2"   "Sst"     "Car4"    "Fst"     "Myl9"    "Nexn"    "Tdgf1"  
 [92] "Dynlrb2" "Anxa2"   "Dkk1"    "Emb"     "Postn"   "Ctla2a"  "Krt19"  
 [99] "H19"     "Aldh1a2"

In [19]:
seurat.list <- lapply(X = seurat.list, FUN = function(x) {
    x <- ScaleData(x, features = features, verbose = FALSE)
    x <- RunPCA(x, features = features, verbose = FALSE)
})

In [ ]:
anchors <- FindIntegrationAnchors(object.list = seurat.list, anchor.features = features, reduction = "rpca")

In [ ]:
# this command creates an 'integrated' data assay
seurat.integrated <- IntegrateData(anchorset = anchors)

In [ ]:
saveRDS(seurat.integrated, paste0(args$outdir, '/seurat_integrated.rds'))

In [ ]:
# specify that we will perform downstream analysis on the corrected data note that the
# original unmodified data still resides in the 'RNA' assay
DefaultAssay(seurat.integrated) <- "integrated"

# Run the standard workflow for visualization and clustering
seurat.integrated <- ScaleData(seurat.integrated, verbose = FALSE)
seurat.integrated <- RunPCA(seurat.integrated, npcs = 30, verbose = FALSE)
seurat.integrated <- RunUMAP(seurat.integrated, reduction = "pca", dims = 1:30, verbose=FALSE)

In [ ]:
DimPlot(seurat.integrated, reduction = "umap", group.by='celltype_extended') + 
    scale_color_manual(values=opts$celltypes_extended.colors) + 
    theme(legend.position='none') 

In [ ]:
DimPlot(seurat.integrated, reduction = "umap", group.by='celltype') + 
    scale_color_manual(values=opts$celltype.colors) + 
    theme(legend.position='none') 

# Transfer

In [34]:
seurat_atlas = seurat.list[[2]]
seurat_query = seurat.list[[1]]

In [37]:
transfer.anchors <- FindTransferAnchors(reference = seurat_atlas, query = seurat_query,
    dims = 1:30, reduction = "rpca")

Performing PCA on the provided reference using 3000 features as input.

Centering and scaling data matrix

Performing PCA on the provided query using 3000 features as input.

Centering and scaling data matrix

Projecting new data onto SVD

Projecting new data onto SVD

Projecting cell embeddings

Finding neighborhoods

Finding anchors

	Found 15038 anchors

Filtering anchors

	Retained 7680 anchors



In [48]:
predictions <- TransferData(anchorset = transfer.anchors, refdata = seurat_atlas$celltype,
    dims = 1:30)
seurat_query <- AddMetaData(seurat_query, metadata = predictions)

Finding integration vectors

Finding integration vector weights

Predicting cell labels



In [20]:
transfer.anchors <- FindTransferAnchors(reference = seurat_atlas, query = seurat_query,
    dims = 1:30, reduction = "pcaproject")

ERROR: Error: No features to use in finding transfer anchors. To troubleshoot, try explicitly providing features to the features parameter and ensure that they are present in both reference and query assays.
